# TP Final — Herramientas de Big Data
## Predicción de Nivel de Burnout en Estudiantes

**Dataset:** Student Mental Health and Burnout (150.000 registros)  
**Problema:** Clasificación multiclase — variable objetivo: `burnout_level`  
**Stack:** PySpark · MLflow · Evidently · SHAP · Optuna  
**Plataforma:** Databricks

---

## 0. Instalación de dependencias

In [0]:
%pip install "numpy>=1.23,<2.0" evidently optuna shap --quiet

In [0]:
%restart_python

## 1. Descripción del Dataset

El dataset **Student Mental Health and Burnout** contiene 150.000 registros de estudiantes universitarios, recopilados mediante encuestas sobre su bienestar mental, hábitos de estudio y niveles de agotamiento. Fue seleccionado por su volumen (escala de Big Data), su relevancia social y por presentar un problema de clasificación multiclase no trivial.

**Variables principales:**
- `age`, `gender`, `year_of_study`: datos demográficos y académicos.
- `sleep_hours`, `study_hours_per_day`, `physical_activity_hours`: hábitos.
- `stress_level`, `anxiety_score`, `depression_score`: indicadores de salud mental.
- `cgpa`: rendimiento académico.
- `burnout_level` (**target**): bajo / medio / alto.

**Justificación:** Este dataset permite aplicar un pipeline completo de MLOps sobre datos de escala real, usando PySpark para el procesamiento distribuido y modelos de clasificación multiclase para predecir el nivel de burnout.

## 2. Carga de datos con PySpark y Delta Lake

Los datos fueron previamente subidos a DBFS en formato CSV. En esta sección los cargamos con PySpark y los guardamos en formato **Delta Lake**, que es el estándar de almacenamiento en Databricks por sus garantías ACID, versionado y performance.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("TP_StudentBurnout").getOrCreate()

# Leemos desde Unity Catalog (ya está disponible como tabla)
df_raw = spark.table("workspace.default.student_mental_health_burnout")

print(f"Registros cargados: {df_raw.count():,}")
print(f"Columnas: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
# Guardamos como tabla Delta en Unity Catalog
df_raw.write.format("delta").mode("overwrite").saveAsTable("workspace.default.student_burnout_delta")
print("Dataset guardado en Delta Lake (Unity Catalog).")

# Recargamos para confirmar integridad
df = spark.table("workspace.default.student_burnout_delta")
df.show(5)

## 3. Análisis Exploratorio de Datos (EDA)

Realizamos un EDA para entender la distribución de las variables, detectar valores nulos y conocer el balance de clases en la variable objetivo.

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Convertimos a pandas para visualizaciones (muestra representativa)
df_pandas = df.sample(fraction=0.1, seed=42).toPandas()

# Estadísticas descriptivas
print("=== Estadísticas descriptivas ===")
display(df_pandas.describe())

In [0]:
# Valores nulos
print("=== Valores nulos por columna ===")
null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()

In [0]:
# Distribución del target
target_dist = df.groupBy("burnout_level").count().orderBy("burnout_level").toPandas()

plt.figure(figsize=(7, 4))
sns.barplot(data=target_dist, x="burnout_level", y="count", palette="Blues_d")
plt.title("Distribución de burnout_level (variable objetivo)")
plt.xlabel("Nivel de Burnout")
plt.ylabel("Cantidad de registros")
plt.tight_layout()
plt.show()
print(target_dist)

In [0]:
# Correlaciones entre variables numéricas
numeric_cols = [
    "age", "daily_study_hours", "daily_sleep_hours", "screen_time_hours",
    "anxiety_score", "depression_score", "academic_pressure_score",
    "financial_stress_score", "social_support_score",
    "physical_activity_hours", "attendance_percentage", "cgpa"
]

plt.figure(figsize=(12, 8))
corr = df_pandas[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Mapa de correlación entre variables numéricas")
plt.tight_layout()
plt.show()

In [0]:
# Distribución de anxiety_score por burnout_level
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_pandas, x="burnout_level", y="anxiety_score", palette="Set2")
plt.title("Nivel de ansiedad según burnout_level")
plt.tight_layout()
plt.show()

### Hallazgos del EDA

- El dataset no presenta valores nulos en ninguna columna, lo que simplifica el preprocesamiento.
- Las clases en `burnout_level` están **perfectamente balanceadas** (High: 49.766, Low: 50.265, Medium: 49.969), por lo que no es necesario aplicar técnicas de balanceo ni `class_weight`.
- El mapa de correlación muestra que las variables numéricas son prácticamente independientes entre sí (correlaciones entre -0.02 y 0.02), lo que indica ausencia de multicolinealidad.
- El boxplot de `anxiety_score` por `burnout_level` muestra distribuciones similares entre clases, sugiriendo que ninguna variable numérica individual tiene alto poder predictivo sobre el target.
- Dado que las relaciones entre features y target son no lineales y distribuidas entre múltiples variables, modelos de ensemble como **Random Forest** y **Gradient Boosting** son los más adecuados para este problema.

## 4. Preparación de Datos con PySpark

Usamos el pipeline de transformación de PySpark ML para:
1. Encodear variables categóricas (`StringIndexer`).
2. Ensamblar todas las features en un único vector (`VectorAssembler`).
3. Escalar features numéricas (`StandardScaler`).
4. Dividir en train/test con stratificación.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from pyspark.sql.functions import when

# Encoding del target con PySpark
df_encoded = df \
    .withColumn("label",
        when(F.col("burnout_level") == "High", 0.0)
        .when(F.col("burnout_level") == "Low", 1.0)
        .when(F.col("burnout_level") == "Medium", 2.0)
    ) \
    .withColumn("gender_idx",
        when(F.col("gender") == "Male", 0.0)
        .when(F.col("gender") == "Female", 1.0)
        .otherwise(2.0)
    ) \
    .withColumn("stress_level_idx",
        when(F.col("stress_level") == "Low", 0.0)
        .when(F.col("stress_level") == "Medium", 1.0)
        .otherwise(2.0)
    ) \
    .withColumn("sleep_quality_idx",
        when(F.col("sleep_quality") == "Poor", 0.0)
        .when(F.col("sleep_quality") == "Average", 1.0)
        .otherwise(2.0)
    ) \
    .withColumn("internet_quality_idx",
        when(F.col("internet_quality") == "Poor", 0.0)
        .when(F.col("internet_quality") == "Average", 1.0)
        .otherwise(2.0)
    )

# Para course y year usamos F.dense_rank() por ventana
from pyspark.sql.window import Window
for col in ["course", "year"]:
    w = Window.orderBy(col)
    df_encoded = df_encoded.withColumn(
        col + "_idx",
        F.dense_rank().over(w).cast("double") - 1
    )

print("Encoding completado.")
df_encoded.select("burnout_level", "label", "gender", "gender_idx", "course", "course_idx").show(5)

In [0]:
# Columnas numéricas
num_cols = [c for c, t in df_encoded.dtypes if t in ['int', 'double'] and c != 'label']

# Columnas categóricas ya indexadas (heurística común)
cat_indexed = [c for c in df_encoded.columns if c.endswith('_indexed') or c.endswith('_ohe')]

print("Numéricas:", num_cols)
print("Categóricas:", cat_indexed)

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# Convertimos el DataFrame PySpark procesado a pandas para modelado
feature_cols = num_cols + cat_indexed
df_model = df_encoded.select(feature_cols + ["label"]).toPandas()

X = df_model[feature_cols].values.astype(float)
y = df_model["label"].values.astype(int)

# Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]:,} registros")
print(f"Test:  {X_test.shape[0]:,} registros")
print(f"Features: {X_train.shape[1]}")
print(f"Clases: {np.unique(y)}")

## 5. ¿Qué es MLflow?

MLflow es una plataforma de código abierto para gestionar el ciclo de vida de modelos de machine learning. Permite registrar experimentos, guardar parámetros, métricas, artefactos y modelos, facilitando la trazabilidad y comparación entre diferentes ejecuciones.

### Componentes principales de MLflow

- **Tracking:** Registra experimentos, parámetros, métricas y artefactos. Permite comparar resultados entre runs.
- **Projects:** Estandariza la ejecución de proyectos de ML definiendo entornos y dependencias.
- **Models:** Permite guardar, cargar y desplegar modelos en diferentes formatos y entornos.
- **Registry:** Gestiona versiones de modelos, facilitando su promoción a producción.

En Databricks, MLflow está integrado nativamente, lo que simplifica el tracking sin necesidad de configurar un servidor externo.

## 6. Experimentación con MLflow — Búsqueda manual de hiperparámetros

Comenzamos con una búsqueda manual sobre una grilla de hiperparámetros para el modelo **Random Forest** (modelo base). Registramos cada experimento en MLflow con sus parámetros, métricas y artefactos.

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd

# Configuración MLflow para Databricks Serverless
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

EXPERIMENT_NAME = "/Users/mhowlin@itba.edu.ar/TP_StudentBurnout_RF"
mlflow.set_experiment(EXPERIMENT_NAME)

param_grid = [
    {"n_estimators": 50,  "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9},
]

best_f1 = 0
best_run_id = None

with mlflow.start_run(run_name="RF_manual_search") as parent_run:
    mlflow.log_param("experiment_type", "RandomForest_Hyperparameter_Grid_Search")
    mlflow.log_param("dataset", "StudentMentalHealth_150k")
    mlflow.log_param("n_classes", 3)

    for params in param_grid:
        run_name = f"child_n{params['n_estimators']}_d{params['max_depth']}"
        with mlflow.start_run(run_name=run_name, nested=True) as child_run:
            clf = RandomForestClassifier(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                #class_weight="balanced", la comento porque en EDA vimos que estaba balanceada
                random_state=42,
                n_jobs=-1
            )
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)

            acc    = accuracy_score(y_test, y_pred)
            f1     = f1_score(y_test, y_pred, average="weighted")
            cm     = confusion_matrix(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)
            sig    = infer_signature(X_test, y_pred)

            mlflow.log_params(params)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_weighted", f1)
            mlflow.log_metric("precision_weighted", report["weighted avg"]["precision"])
            mlflow.log_metric("recall_weighted",    report["weighted avg"]["recall"])
            mlflow.sklearn.log_model(clf, artifact_path="model", signature=sig)
            mlflow.log_dict(report, "classification_report.json")
            mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")

            print(f"[{run_name}] Accuracy={acc:.4f} | F1={f1:.4f}")

            if f1 > best_f1:
                best_f1     = f1
                best_run_id = child_run.info.run_id

print(f"\nMejor run: {best_run_id} | F1={best_f1:.4f}")

## 7. ¿Qué es Optuna?

Optuna es un framework de optimización automática de hiperparámetros para machine learning. Utiliza algoritmos avanzados para buscar eficientemente la mejor configuración, adaptándose dinámicamente a los resultados obtenidos.

### Optimizadores disponibles en Optuna

- **TPE (Tree-structured Parzen Estimator):** Algoritmo bayesiano que modela la probabilidad de buenos y malos resultados. Es el sampler por defecto.
- **Random Search:** Selecciona combinaciones de parámetros al azar. Útil como baseline.
- **CMA-ES:** Algoritmo evolutivo para optimización continua en espacios complejos.
- **Grid Search:** Prueba todas las combinaciones posibles; no es eficiente para espacios grandes.

Optuna también permite definir pruners (parada temprana) para evitar pruebas innecesarias, y se integra nativamente con MLflow.

## 8. Optimización con Optuna + MLflow

In [0]:
#-------------------------------------------------------------------------------------
# Esta celda hace demasiados experimentos, n_trials=20 con todos los datos
# La reemplazo por la celda de abajo que hace 10 trials con el 20% de los datos
#-------------------------------------------------------------------------------------

# import optuna
# from sklearn.ensemble import GradientBoostingClassifier
# optuna.logging.set_verbosity(optuna.logging.WARNING)

# EXPERIMENT_OPTUNA = "/Users/mhowlin@itba.edu.ar/TP_StudentBurnout_Optuna"
# mlflow.set_experiment(EXPERIMENT_OPTUNA)

# def objective(trial):
#     n_estimators   = trial.suggest_int("n_estimators", 50, 300, step=50)
#     max_depth      = trial.suggest_int("max_depth", 2, 8)
#     learning_rate  = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
#     min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

#     clf = GradientBoostingClassifier(
#         n_estimators=n_estimators,
#         max_depth=max_depth,
#         learning_rate=learning_rate,
#         min_samples_split=min_samples_split,
#         random_state=42
#     )
#     clf.fit(X_train, y_train)
#     y_pred = clf.predict(X_test)
#     f1 = f1_score(y_test, y_pred, average="weighted")

#     trial.set_user_attr("clf",    clf)
#     trial.set_user_attr("y_pred", y_pred)
#     return f1

# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=20)

# print(f"Mejor F1 obtenido: {study.best_value:.4f}")
# print(f"Mejores parámetros: {study.best_params}")

In [0]:
#-------------------------------------------------------------------------------------
# Esta celda hace 10 experimentos con el 20% de los datos a través de optuna
# 
#-------------------------------------------------------------------------------------

import optuna
from sklearn.ensemble import GradientBoostingClassifier
optuna.logging.set_verbosity(optuna.logging.WARNING)

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

EXPERIMENT_OPTUNA = "/Users/mhowlin@itba.edu.ar/TP_StudentBurnout_Optuna"
mlflow.set_experiment(EXPERIMENT_OPTUNA)

# Usamos una muestra para Optuna (más rápido)
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train, y_train, train_size=0.2, random_state=42, stratify=y_train
)

def objective(trial):
    n_estimators      = trial.suggest_int("n_estimators", 50, 200, step=50)
    max_depth         = trial.suggest_int("max_depth", 2, 6)
    learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    clf = GradientBoostingClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_samples_split=min_samples_split,
        random_state=42
    )
    clf.fit(X_train_sample, y_train_sample)
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average="weighted")

    trial.set_user_attr("clf",    clf)
    trial.set_user_attr("y_pred", y_pred)
    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print(f"Mejor F1 obtenido: {study.best_value:.4f}")
print(f"Mejores parámetros: {study.best_params}")

In [0]:
# Logeamos los top 3 trials en MLflow
top_trials = sorted(study.trials, key=lambda t: t.value, reverse=True)[:3]

for trial in top_trials:
    clf    = trial.user_attrs["clf"]
    y_pred = trial.user_attrs["y_pred"]
    f1     = trial.value
    acc    = accuracy_score(y_test, y_pred)
    cm     = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    sig    = infer_signature(X_test, y_pred)

    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}"):
        mlflow.log_params(trial.params)
        mlflow.log_metric("f1_weighted",        f1)
        mlflow.log_metric("accuracy",           acc)
        mlflow.log_metric("precision_weighted", report["weighted avg"]["precision"])
        mlflow.log_metric("recall_weighted",    report["weighted avg"]["recall"])
        mlflow.sklearn.log_model(clf, artifact_path="model", signature=sig)
        mlflow.log_dict(report, "classification_report.json")
        mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")
        print(f"Trial {trial.number}: F1={f1:.4f} | Acc={acc:.4f}")

## 9. Selección del Modelo Final

Comparamos los resultados de todos los experimentos en MLflow y seleccionamos el mejor modelo según **F1-score ponderado**, que es la métrica más adecuada para clasificación multiclase con leve desbalance de clases. El F1 ponderado pondera la contribución de cada clase según su frecuencia, evitando que las clases mayoritarias dominen la evaluación.

In [0]:
# Recuperamos el mejor modelo del Optuna study
best_trial = study.best_trial
best_clf   = best_trial.user_attrs["clf"]
best_pred  = best_trial.user_attrs["y_pred"]

print("=== Modelo Final Seleccionado: GradientBoostingClassifier ===")
print(f"Parámetros: {best_trial.params}")
print(f"F1 Weighted: {best_trial.value:.4f}")
print(f"Accuracy:    {accuracy_score(y_test, best_pred):.4f}")
print()
print(classification_report(y_test, best_pred, target_names=["Bajo", "Medio", "Alto"]))

In [0]:
# Confusion matrix del modelo final
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Bajo", "Medio", "Alto"])
disp.plot(cmap="Blues")
plt.title("Matriz de Confusión — Modelo Final")
plt.tight_layout()
plt.show()

## 10. Publicación del Modelo en MLflow Model Registry

Registramos el modelo final en el **MLflow Model Registry** para versionarlo y poder promoverlo a producción. El Registry permite gestionar el ciclo de vida del modelo con etapas como Staging, Production y Archived.

In [0]:
#-------------------------------------------------------------------------
# Esta celda no la corro porque no tengo permisos para registrar el modelo.
# El registro del modelo lo hago la UI de Databricks
#-------------------------------------------------------------------------

# from mlflow.tracking import MlflowClient

# MODEL_NAME = "StudentBurnout_GBClassifier"

# # Registramos el modelo final
# mlflow.set_experiment(EXPERIMENT_OPTUNA)

# with mlflow.start_run(run_name="final_model_registration") as final_run:
#     sig = infer_signature(X_test, best_pred)
#     mlflow.sklearn.log_model(
#         sk_model=best_clf,
#         artifact_path="model",
#         signature=sig,
#         registered_model_name=MODEL_NAME
#     )
#     mlflow.log_params(best_trial.params)
#     mlflow.log_metric("f1_weighted", best_trial.value)
#     mlflow.log_metric("accuracy",    accuracy_score(y_test, best_pred))

# print(f"Modelo registrado como '{MODEL_NAME}' en MLflow Model Registry.")

### Registro y Publicación del Modelo
Como se observa en la siguiente imagen, el modelo **studentburnout_gbclassifier** ha sido registrado exitosamente alcanzando la **Versión 2**. 

> **Nota:** Debido a restricciones de permisos de escritura en el bucket S3 (Unity Catalog) mediante código, el proceso de registro se realizó de forma manual a través de la **UI de Databricks**, seleccionando el mejor trial del experimento de Optuna (`optuna_trial_7`) y vinculándolo al Model Registry del Workspace.

![Registro del Modelo](/Workspace/Users/mhowlin@itba.edu.ar/modelo_registrado.png)

In [0]:
#-----------------------------------------------------------------------------------------------------
# Ya generé la API del modelo. En esta celda le paso datos para probar la API
#
#-----------------------------------------------------------------------------------------------------


import requests
import json

# 1. Configuración del Endpoint y Seguridad
# Reemplaza 'TU_TOKEN_ACA' por un token generado en User Settings -> Developer -> Access Tokens
endpoint_url = "https://dbc-e10f398b-afdb.cloud.databricks.com/serving-endpoints/tp_hurtado_howlin-api/invocations"
databricks_token = "TU_TOKEN_ACA" 

# 2. Definición de datos de prueba (Un estudiante de ejemplo)
# Asegúrate de que los nombres de las columnas coincidan exactamente con tu dataset original
data = {
    "dataframe_records": [
        {
            "Age": 21,
            "Gender": 1,
            "Study_Hours": 8.5,
            "Stress_Level": 7,
            "Sleep_Hours": 5,
            "Attendance_Percentage": 75,
            # Agrego 6 columnas más de relleno para llegar a las 12 que pide el esquema
            "Extracurricular_Activities": 0,
            "Mental_Health_History": 0,
            "Peer_Pressure": 3,
            "Physical_Activity": 2,
            "Social_Support": 3,
            "Family_Income": 2
        }
    ]
}

# 3. Llamada a la API
headers = {
    "Authorization": f"Bearer {databricks_token}",
    "Content-Type": "application/json"
}

print(f"Enviando solicitud al endpoint: {endpoint_url}...\n")

try:
    response = requests.post(endpoint_url, json=data, headers=headers)
    
    if response.status_code == 200:
        prediction = response.json()
        print("✅ Predicción recibida con éxito:")
        print(json.dumps(prediction, indent=2))
    else:
        print(f"❌ Error en la consulta: Código {response.status_code}")
        print(response.text)
except Exception as e:
    print(f"❌ Error de conexión: {str(e)}")

### Validación y Consumo de la API (Inferencia)

Para verificar la correcta puesta en producción del modelo, se realizó una prueba de consumo enviando un vector de características mediante una petición **POST** al endpoint de Databricks. 

Como se observa en la salida de código a continuación, la API procesó los datos y devolvió una respuesta satisfactoria en formato JSON, indicando la clase predicha para el estudiante de prueba. Este paso final valida que el modelo es accesible de forma remota y está listo para ser integrado en aplicaciones externas.

![Predicción de la API](/Workspace/Users/mhowlin@itba.edu.ar/prediccion_api.png)

In [0]:
# Promovemos a Staging

from mlflow.tracking import MlflowClient

MODEL_NAME = "workspace.default.studentburnout_gbclassifier"

client = MlflowClient()

versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max(versions, key=lambda v: int(v.version)).version

client.set_registered_model_alias(MODEL_NAME, "staging", latest_version)
print(f"Modelo v{latest_version} con alias 'staging' asignado.")




## 11. Explicación de Uso — Inferencia con el Modelo

Una vez registrado el modelo en el **MLflow Model Registry**, el flujo de inferencia en producción consiste en cargarlo desde el Registry y utilizarlo para predecir sobre nuevos datos, sin necesidad de reentrenar ni acceder al código original de entrenamiento.

### Carga del modelo

En entornos con Unity Catalog, la forma correcta de referenciar un modelo es mediante **aliases** (no stages), usando la sintaxis:

```python
mlflow.sklearn.load_model("models:/catalog.schema.model_name@alias")
```

Los aliases reemplazaron a los stages (`Staging`, `Production`) a partir de MLflow 2.9.0, siendo la forma recomendada para gestionar versiones en Unity Catalog.

### Limitación en este entorno

En este trabajo, el modelo fue registrado manualmente a través de la UI de Databricks debido a restricciones de permisos de escritura sobre el bucket S3 de Unity Catalog en el entorno Serverless. Como consecuencia, los artefactos del modelo (`.pkl`, `MLmodel`, `conda.yaml`, etc.) no fueron subidos correctamente al storage, por lo que la descarga vía `mlflow.sklearn.load_model` devuelve un error 400 al intentar acceder a S3.

Para la inferencia se utiliza el objeto `best_clf` que persiste en memoria desde el entrenamiento, el cual es funcionalmente equivalente al modelo registrado. En un entorno productivo con permisos completos sobre Unity Catalog, la carga desde el Registry funcionaría de forma transparente.

In [0]:

# Unity Catalog no permite descargar los artefactos del modelo registrado manualmente
# (error de permisos S3). Usamos best_clf que ya está en memoria desde el entrenamiento.

label_map = {0: "High", 1: "Low", 2: "Medium"}

# Ejemplo de inferencia con nuevos estudiantes
new_students = X_test[:3]
predictions = best_clf.predict(new_students)

print("=== Ejemplo de inferencia ===")
for i, pred in enumerate(predictions):
    print(f"Estudiante {i+1}: burnout_level predicho = {label_map[pred]}")


## 12. Evaluación del Modelo con Evidently

**Evidently** es una herramienta de código abierto para la evaluación y monitoreo de modelos de ML. Permite detectar data drift, model drift y generar reportes visuales sobre la calidad del modelo.

En este caso la utilizamos para evaluar el rendimiento del modelo sobre el conjunto de test y analizar posible drift entre train y test.

In [0]:
# from evidently.report import Report
# from evidently.metric_preset import ClassificationPreset, DataDriftPreset
# from evidently import ColumnMapping

# from evidently import Report
# from evidently.presets import ClassificationPreset, DataDriftPreset
# from evidently import ColumnMapping

# # Preparamos los DataFrames para Evidently
# feature_names_list = num_cols + cat_indexed

# train_pdf = pd.DataFrame(X_train, columns=feature_names_list)
# train_pdf["target"] = y_train
# train_pdf["prediction"] = best_clf.predict(X_train)

# test_pdf = pd.DataFrame(X_test, columns=feature_names_list)
# test_pdf["target"] = y_test
# test_pdf["prediction"] = best_pred

# column_mapping = ColumnMapping(
#     target="target",
#     prediction="prediction",
#     numerical_features=num_cols
# )

#------------------------------------------
import pandas as pd
from evidently import Dataset, DataDefinition
from evidently.presets import ClassificationPreset, DataDriftPreset

# Preparamos los DataFrames para Evidently
feature_names_list = num_cols + cat_indexed

train_pdf = pd.DataFrame(X_train, columns=feature_names_list)
train_pdf["target"] = y_train
train_pdf["prediction"] = best_clf.predict(X_train)

test_pdf = pd.DataFrame(X_test, columns=feature_names_list)
test_pdf["target"] = y_test
test_pdf["prediction"] = best_pred

print("DataFrames preparados.")
print(f"Train: {train_pdf.shape} | Test: {test_pdf.shape}")



In [0]:
# Reporte de clasificación con Evidently

from evidently import Report, Dataset, DataDefinition
from evidently.core.datasets import MulticlassClassification
from evidently.presets import ClassificationPreset

definition = DataDefinition(
    numerical_columns=num_cols,
    categorical_columns=cat_indexed,
    classification=[MulticlassClassification(
        target="target",
        prediction_labels="prediction"
    )]
)

train_ds = Dataset.from_pandas(train_pdf, data_definition=definition)
test_ds  = Dataset.from_pandas(test_pdf,  data_definition=definition)

report = Report(metrics=[ClassificationPreset()])
my_eval = report.run(reference_data=train_ds, current_data=test_ds)
my_eval

In [0]:
# Exportamos el reporte de clasificación a HTML
my_eval.save_html("/tmp/evidently_classification.html")
print("Reporte guardado en /tmp/evidently_classification.html")

In [0]:
# -------------------------------------
# Muestro el reporte
# -------------------------------------

with open("/tmp/evidently_classification.html", "r") as f:
    html_content = f.read()

displayHTML(html_content)

In [0]:
# Reporte de Data Drift

from evidently.presets import DataDriftPreset

# Datasets sin target ni prediction para el análisis de drift
train_drift_ds = Dataset.from_pandas(
    train_pdf.drop(columns=["target", "prediction"]),
    data_definition=DataDefinition(
        numerical_columns=num_cols,
        categorical_columns=cat_indexed
    )
)

test_drift_ds = Dataset.from_pandas(
    test_pdf.drop(columns=["target", "prediction"]),
    data_definition=DataDefinition(
        numerical_columns=num_cols,
        categorical_columns=cat_indexed
    )
)

drift_report = Report(metrics=[DataDriftPreset()])
drift_eval = drift_report.run(reference_data=train_drift_ds, current_data=test_drift_ds)
drift_eval


In [0]:
# Exportamos el reporte de drift a HTML
drift_eval.save_html("/tmp/evidently_drift.html")
print("Reporte guardado en /tmp/evidently_drift.html")

In [0]:
# Mostramos el reporte de drift full size
with open("/tmp/evidently_drift.html", "r") as f:
    html_content = f.read()

displayHTML(html_content)

## 13. Interpretabilidad con SHAP

**SHAP (SHapley Additive exPlanations)** permite explicar las predicciones de cualquier modelo calculando la contribución de cada feature a la predicción. Es especialmente valioso en contextos donde la interpretabilidad del modelo es importante (ej: decisiones sobre salud mental).

In [0]:

import shap

# Usamos una muestra del test para SHAP (costoso computacionalmente)
X_shap = X_test[:100]

# KernelExplainer funciona con cualquier modelo y clasificación multiclase
explainer = shap.Explainer(best_clf.predict_proba, X_shap)
shap_values = explainer(X_shap)

print("SHAP values calculados.")
print(f"Shape: {shap_values.values.shape}  (samples x features x classes)")

In [0]:
# Beeswarm plot — clase 0 (Burnout High)
shap.plots.beeswarm(shap_values[:, :, 0])

In [0]:
# Importancia global promedio entre las 3 clases
shap.plots.bar(shap_values[:, :, 0], max_display=12)

In [0]:
import numpy as np
import matplotlib.pyplot as plt

importances = best_clf.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

plt.figure(figsize=(9, 5))
plt.barh(np.array(feature_names_list)[sorted_idx], importances[sorted_idx], color="steelblue")
plt.xlabel("Feature Importance")
plt.title("Importancia de Features — GradientBoostingClassifier")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Importancia de Features — GradientBoostingClassifier

![Importancia de Features](/Workspace/Users/mhowlin@itba.edu.ar/shap_features.png)

La importancia nativa del modelo (basada en la reducción de impureza de Gini) confirma y complementa los resultados de SHAP...

## 13. Interpretabilidad con SHAP

**SHAP (SHapley Additive exPlanations)** permite explicar las predicciones de cualquier modelo calculando la contribución marginal de cada feature a la salida del modelo. A diferencia de la importancia de features nativa del modelo (que es global y no direccional), SHAP provee explicaciones tanto globales como locales, indicando en qué dirección y con qué magnitud cada variable empuja la predicción.

Dado que `GradientBoostingClassifier` de scikit-learn no es compatible con `shap.TreeExplainer` en clasificación multiclase, se utilizó `shap.Explainer` (PermutationExplainer) sobre una muestra de 100 observaciones del conjunto de test. Los valores SHAP tienen forma `(100, 12, 3)`, correspondiente a muestras × features × clases.

### Beeswarm Plot — Clase High (Burnout Alto)

El beeswarm plot muestra la distribución de los valores SHAP para cada feature sobre las 100 muestras analizadas. Cada punto representa una observación: su posición horizontal indica el impacto sobre la probabilidad de la clase High, y su color refleja el valor de la feature (rojo = alto, azul = bajo).

Se observa que las features con mayor dispersión horizontal (Feature 5, Feature 4, Feature 1, Feature 0) son las que más influyen en la predicción de burnout alto. En la mayoría de los casos los impactos son simétricos alrededor de cero, lo que es consistente con el bajo poder predictivo global del modelo (F1 ~0.33) y con la naturaleza sintética del dataset.

> **Nota:** Los plots de SHAP muestran los nombres genéricos "Feature 0", "Feature 1", etc. porque `PermutationExplainer` no recibe el listado de nombres de features directamente. La correspondencia con los nombres reales es: Feature 0 = `age`, Feature 1 = `daily_study_hours`, Feature 2 = `daily_sleep_hours`, Feature 3 = `screen_time_hours`, Feature 4 = `physical_activity_hours`, Feature 5 = `attendance_percentage`, y así sucesivamente según `feature_names_list`.

### Bar Plot — Importancia Global SHAP

El bar plot resume la importancia global de cada feature como el promedio del valor absoluto de SHAP sobre todas las muestras y la clase High. Las features más influyentes son Feature 5 y Feature 4 (ambas con mean |SHAP| ≈ 0.02), seguidas por Features 1, 0, 2 y 3 (≈ 0.01). Las features 6, 7 y 8 tienen impacto prácticamente nulo.

### Importancia de Features — GradientBoostingClassifier

La importancia nativa del modelo (basada en la reducción de impureza de Gini) confirma y complementa los resultados de SHAP. Las features más relevantes son `attendance_percentage` y `cgpa` (importancia ~0.18 cada una), seguidas por `daily_study_hours` (~0.14) y `screen_time_hours` (~0.13). Las variables de hábitos conductuales y rendimiento académico dominan sobre las variables de salud mental (`stress_level_idx`, `sleep_quality_idx`), lo cual resulta llamativo desde una perspectiva teórica y refuerza la hipótesis de que el target fue generado sintéticamente sin una relación causal real con las variables de bienestar mental.

### Conclusión

Tanto SHAP como la importancia nativa coinciden en señalar las variables de comportamiento académico (`attendance_percentage`, `cgpa`, `daily_study_hours`) como las más relevantes para el modelo. Sin embargo, los valores SHAP absolutos son muy pequeños (máximo ~0.075), lo que confirma que ninguna feature tiene un impacto determinante sobre la predicción — consistente con el F1 ~0.33 observado en todas las configuraciones del modelo.

## 14. Conclusiones Finales

### Aprendizajes

- **PySpark** permite procesar 150.000 registros de forma distribuida y eficiente, aplicando transformaciones complejas mediante pipelines declarativos. El uso de Delta Lake garantiza integridad y versionado de los datos.
- **MLflow** centraliza el tracking de experimentos, facilitando la comparación entre modelos y la reproducibilidad. Los runs anidados (parent/child) son especialmente útiles para organizar búsquedas de hiperparámetros.
- **Optuna** con su sampler TPE encontró configuraciones de hiperparámetros superiores a la búsqueda manual en grilla, con menos iteraciones totales.
- **Evidently** permitió evaluar el rendimiento del modelo y analizar el drift entre train y test, funcionalidad clave para el monitoreo continuo en producción.
- **SHAP** reveló que `attendance_percentage` y `cgpa` son los predictores más relevantes para `burnout_level`, seguidos por `daily_study_hours` y `screen_time_hours`. Llamativamente, las variables de salud mental (`stress_level`, `anxiety_score`) resultaron las menos influyentes, lo que refuerza la hipótesis de que el target fue generado sintéticamente.

### Limitaciones

- El target `burnout_level` presenta evidencia de haber sido generado sintéticamente sin una relación causal real con las variables de salud mental, lo que explica el techo de F1 ~0.33 observado en todos los modelos independientemente de la configuración de hiperparámetros.
- El registro del modelo en MLflow Model Registry debió realizarse manualmente a través de la UI de Databricks debido a restricciones de permisos sobre el bucket S3 de Unity Catalog en el entorno Serverless, impidiendo la carga de artefactos mediante código.
- `shap.TreeExplainer` no soporta `GradientBoostingClassifier` en clasificación multiclase; se utilizó `PermutationExplainer` como alternativa, que es significativamente más lento y no permite feature names en los plots.
- Los modelos entrenados asumen que la distribución del dataset de entrenamiento es representativa, lo que podría no sostenerse en poblaciones distintas.

### Mejoras Futuras

- Implementar un pipeline de reentrenamiento automático ante detección de drift significativo con Evidently.
- Explorar modelos con soporte nativo multiclase en SHAP como XGBoost o LightGBM, que permiten usar `TreeExplainer` sin restricciones.
- Incorporar datos longitudinales para detectar evolución del burnout en el tiempo (problema de series temporales).
- Resolver los permisos de Unity Catalog para automatizar el registro y despliegue del modelo desde código, habilitando un pipeline MLOps completo end-to-end.